# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MaryumAkram16/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Type: Scoring / Ranking (implemented as binary classification underneath).**

The real deliverable for Lane 2 is a ranked list — "review these pages first" — not a single yes/no label in isolation. Under the hood, I'll train a binary classifier (`is_declining_label`) and use its predicted probability as the ranking score, but what matters is the *order* of the top 20-50 pages, not whether every individual prediction is correct. That's what makes it a scoring/ranking task rather than plain classification: the output is consumed as a sorted queue with limited "read depth," not as independent per-row verdicts.

In [11]:
# Quick check: how skewed is the label? This affects why ranking (not raw accuracy) is the right frame.
import pandas as pd

url = "https://raw.githubusercontent.com/MaryumAkram16/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

print(df["trend_direction"].value_counts())
print(f"\n'down' share: {(df['trend_direction'] == 'down').mean():.1%}")

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

'down' share: 54.2%


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target:** `is_declining_label = (trend_direction == "down")`

**Where it comes from:** This is a *defined rule*, not a directly observed future outcome. `trend_direction` is itself a derived bucket computed from `trend_pct` (the percentage change over the current 90-day window), so the label describes a page's *current* trend state, not something that happens after a decision point. This makes it a **proxy label** — useful for building the workflow end-to-end now, but a stronger capstone label would be forward-looking (e.g., features from the prior 90 days → decline in the next 30 days), so a client's current outcome isn't leaking into its own label.

In [12]:
label = (df["trend_direction"] == "down").astype(int)
print(f"Label distribution:\n{label.value_counts()}")
print(f"\nPositive rate: {label.mean():.1%}")


Label distribution:
trend_direction
1    16262
0    13738
Name: count, dtype: int64

Positive rate: 54.2%


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: Precision@50.**

A content team can only realistically review a fixed number of pages per week (roughly 20-50, per my ML-02 framing). Precision@50 answers exactly the question that matters to them: "of the top 50 pages the system tells me to look at first, how many are actually worth it?" Generic accuracy or ROC AUC would reward the model for getting the easy, low-stakes majority right, but say nothing about whether the specific handful of pages a human will actually open are good picks. This is also the metric the starter pipeline already reports, so it's directly comparable: baseline rule = 0.240, random forest = 0.740 (from `docs/ml-intern-dataset-and-lane-guide.md`).

As a sanity check, ranking purely by raw `impressions_90d` already gets Precision@50 = 0.420 — better than the hand-tuned baseline rule's 0.240, but still well short of the random forest's 0.740. That gap between a single strong feature and the full model is exactly what justifies going past a fixed rule.

In [13]:
def precision_at_k(scores, labels, k):
    order = scores.sort_values(ascending=False).index
    top_k_labels = labels.loc[order][:k]
    return top_k_labels.mean()

# Demo using a naive score (impressions_90d) just to show the metric mechanics
naive_score = df["impressions_90d"]
p50 = precision_at_k(naive_score, label, 50)
print(f"Precision@50 using impressions_90d alone as the score: {p50:.3f}")
print("(For comparison: starter baseline rule = 0.240, random forest = 0.740)")


Precision@50 using impressions_90d alone as the score: 0.420
(For comparison: starter baseline rule = 0.240, random forest = 0.740)


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one page (`content_id`)**, for one client (`client_id`), summarizing that page's last 90 days of observed search/content signals. This matches the starter dataset's grain exactly — it is not one row per day or per query, so trends like `trend_direction` are already pre-aggregated into a single snapshot per page.

In [14]:
unit_cols = ["content_id", "client_id", "impressions_90d", "avg_position",
             "ctr", "content_age_days", "days_since_last_update",
             "word_count", "trend_direction"]
print(f"Rows (pages): {len(df)}")
print(f"Unique content_id values: {df['content_id'].nunique()}")
df[unit_cols].head(5)

Rows (pages): 30000
Unique content_id values: 30000


,content_id,client_id,impressions_90d,avg_position,ctr,content_age_days,days_since_last_update,word_count,trend_direction
0,content_304f48230142,client_f369cb89fc,3803,10.6,0.76,187,20,3221.0,down
1,content_a1fb4e703a9e,client_4e07408562,15320,20.3,0.05,445,25,2481.0,down
2,content_9aa793d4d895,client_7f2253d7e2,12581,36.5,0.09,141,20,3515.0,down
3,content_331d6c4de07b,client_19581e27de,11751,6.2,0.49,463,22,NaN,stable
4,content_d99b7a2d90ca,client_3fdba35f04,19140,44.0,0.13,263,14,2803.0,down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule can only combine a small, hand-picked set of thresholds (e.g., "stale AND declining AND low impressions"). But the signals that actually separate declining from stable pages interact in ways that aren't obvious to hand-code — a page's freshness might matter a lot at high impression volume but barely at all at low volume, or a mid-range CTR might be a red flag only when combined with a strong position. A model can discover these interactions automatically instead of a human guessing which combinations matter and at what cutoffs. The evidence for this is already in hand: on this exact dataset and label, the hand-written baseline rule scored Precision@50 = 0.240, while a random forest scored 0.740 — roughly three times as many correct picks in the same top-50 slots, with no new data, just a model finding structure a fixed rule missed.

(Note: the `high` impression tier has no `flat`-trend pages in this sample — hence the NaN — which is itself informative: high-visibility pages rarely sit still, they're either declining or growing.)

In [15]:
# Illustrate one such interaction: does the "low CTR" signal mean the same thing
# at different impression levels? A fixed single-threshold rule can't adapt to this.
df["impression_tier"] = pd.qcut(df["impressions_90d"], q=4, labels=["low", "mid-low", "mid-high", "high"], duplicates="drop")
ctr_by_tier_and_trend = df.groupby(["impression_tier", "trend_direction"])["ctr"].mean().unstack()
print(ctr_by_tier_and_trend.round(4))


trend_direction    down    flat     new  stable      up
impression_tier                                        
low              0.7722  1.4277  1.4137  2.1919  1.6132
mid-low          0.2095  0.2321  0.2745  0.2914  0.2731
mid-high         0.2117  0.0183  0.2095  0.2565  0.2618
high             0.2726     NaN  0.2060  0.3721  0.3323


/tmp/ipykernel_1867/4187680935.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  ctr_by_tier_and_trend = df.groupby(["impression_tier", "trend_direction"])["ctr"].mean().unstack()


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.